# Biber Daten analysieren

## Daten importieren

Pandas-Modul laden.

In [0]:
import pandas as pd

Dateien einlesen

In [0]:
df = pd.read_csv("../Daten/Biber-Daten-Long.anonym.csv")

oder für komprimierte Rohdaten-Dateien

In [0]:
df = pd.read_csv("../Daten/Biber-Daten-Long.anonym.csv.gz", compression='gzip')

## Überblick verschaffen

In [0]:
df.head()

In [0]:
df.tail(3)

In [0]:
df.describe()

In [0]:
df.info()

## Vorbereitung

Bonuspunkte herausfiltern

In [0]:
f = df['Aufgabe'] != 'Bonus'
df = df[f]

Teams markieren

In [0]:
df["Team"] = df["Teamname"] != "none"

## Daten überprüfen

In [0]:
df.info()

Modul für interaktive Tabellen laden und konfigurieren.

In [0]:
import itables
itables.options.maxBytes = "5MB"

In [0]:
itables.show(df)

## Daten mit groupby oder pivot_table analysieren

In [0]:
df[['Jahr', 'Klassenstufe', 'Punkte']]\
  .groupby(['Jahr', 'Klassenstufe'])\
  .mean()

In [0]:
df[["Jahr", "Klassenstufe", "Schule", "Punkte"]]\
    .groupby(["Jahr", "Schule", "Klassenstufe"])\
    .mean()\
    .sort_values(["Jahr", "Punkte"], ascending=[True, False])

In [0]:
df.pivot_table(
    index = ['Klassenstufe', 'Schule'],
    columns = ['Jahr', 'Team'],
    values='Punkte',
    aggfunc='mean'
)

In [0]:
aufgabenauswahl =   (df['Aufgabe'] == 'Schichte nach Dichte!') \
                  | (df['Aufgabe'] == 'SOS aus den Bergen')

df[aufgabenauswahl]\
  .groupby( ['Jahr', 'Aufgabe', 'Klassenstufe'] )['Punkte']\
  .agg(
    count    = lambda x: x.count(),
    min      = lambda x: x.min(),
    q01      = lambda x: x.quantile(0.01),
    q05      = lambda x: x.quantile(0.05),
    q25      = lambda x: x.quantile(0.25),
    q50      = lambda x: x.quantile(0.50),
    q75      = lambda x: x.quantile(0.75),
    q95      = lambda x: x.quantile(0.95),
    q99      = lambda x: x.quantile(0.99),
    max      = lambda x: x.max(),
    mean     = lambda x: round(x.mean(), 1),
    std      = lambda x: round(x.std(), 1),
    kurtosis = lambda x: round(x.kurtosis(), 1),
    sum      = lambda x: x.sum(),
  )\
  .sort_values(['Jahr', 'Aufgabe', 'Klassenstufe'], ascending=[True, True, True])

In [0]:
schwierigste = True
anzahl_aufgaben = 26
nur_klassenstufe = (df['Klassenstufe'] == '9+10') | (df['Klassenstufe'] == '11+12+13')

itables.show(
  df[nur_klassenstufe]\
  .groupby( ['Aufgabe', 'Jahr', 'Klassenstufe'] )['Punkte']\
  .agg(
    antworten             = lambda x: x.count(),
    min                   = lambda x: x.min(),
    max                   = lambda x: x.max(),
    unbeantwortet         = lambda x: x[x == 0].count(),
    falsch                = lambda x: x[x == x.min()].count() if x.min() < 0 else 0,
    richtig               = lambda x: x[x == x.max()].count() if x.max() > 0 else 0,
    unbeantwortet_prozent = lambda x: round(x[x == 0].count() / x.count() * 100, 1),
    falsch_prozent        = lambda x: round(x[x == x.min()].count() / x.count() * 100 if x.min() < 0 else 0, 1),
    richtig_prozent       = lambda x: round(x[x == x.max()].count() / x.count() * 100 if x.max() > 0 else 0, 1)
  )\
  .sort_values(['richtig_prozent','falsch_prozent', 'min', 'Aufgabe'], ascending=[schwierigste, not schwierigste, schwierigste, True])\
  .head(anzahl_aufgaben)
)

In [0]:
anzahl = 3
calculate_worst_scores = False

punkte = (
    df.groupby(
        ["Jahr", "Klassenstufe", "Schule", "Klasse", "Name"], as_index=False
    )["Punkte"]
    .sum()
)

punkte["Rang"] = (
    punkte.groupby(["Jahr", "Klassenstufe"])["Punkte"]
    .rank(method="min", ascending=calculate_worst_scores)
)

rangliste = (
    punkte[punkte["Rang"] <= anzahl]
    [["Jahr", "Klassenstufe", "Schule", "Klasse", "Name", "Punkte"]]
    .sort_values(
        ["Jahr", "Klassenstufe", "Punkte", "Klasse", "Name"],
        ascending=[True, False, calculate_worst_scores, True, True]
    )
    .reset_index(drop = True)
)

rangliste.tail(10)

In [0]:
rangliste[rangliste['Jahr'] == 2025]

## Grafiken erstellen

In [0]:
df.plot(kind="box", column="Punkte", by=['Jahr', 'Klassenstufe'], showmeans=True, rot=90)